In [ ]:
import h5py
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

Dataset_files = [
    "H-H1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_16KHZ_R1-1187008867-32.hdf5"
    "H-H1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187006835-4096.hdf5"
]

def load_strain(file_path):
    with h5py.File(file_path, "r") as f:
        strain = f["strain"]["Strain"][:]
    return strain

WINDOW_SIZE = 4096
STEP_SIZE = 2048

X = []
y = []

def generate_windows(signal, label):

    for i in range(
        0,
        len(signal) - WINDOW_SIZE,
        STEP_SIZE
    ):

        segment = signal[i:i + WINDOW_SIZE]

        X.append(segment)
        y.append(label)

for file in positive_files:
    signal = load_strain(file)
    generate_windows(signal, 1)

for file in negative_files:
    signal = load_strain(file)
    generate_windows(signal, 0)

X = np.array(X)
y = np.array(y)

print("Dataset Shape:", X.shape)

scaler = StandardScaler()

X = scaler.fit_transform(X)

X = X.reshape(
    X.shape[0],
    X.shape[1],
    1
)

print("RNN Input Shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Sequential()

model.add(
    SimpleRNN(
        128,
        activation='tanh',
        return_sequences=True,
        input_shape=(
            X_train.shape[1],
            1
        )
    )
)

model.add(
    Dropout(0.3)
)

model.add(
    SimpleRNN(
        64,
        activation='tanh'
    )
)

model.add(
    Dropout(0.3)
)

model.add(
    Dense(
        64,
        activation='relu'
    )
)

model.add(
    Dense(
        32,
        activation='relu'
    )
)

model.add(
    Dense(
        1,
        activation='sigmoid'
    )
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", accuracy)

predictions = model.predict(X_test)

predictions = (
    predictions > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        predictions
    )
)

model.save(
    "RNN_GW_Merger_Detector.h5"
)

print("Model Saved Successfully")